# 02: Embedding Extraction — Extract and Cache from All Models

**Goal**: Extract embeddings from SBERT and CLIP (text-only) on the VSR dataset and cache results.

This is the most computationally expensive step — results are cached to disk as .npy files.

All embeddings are L2-normalized before saving.

**Note**: This notebook can run locally on CPU or GPU. For faster extraction, use Google Colab with GPU acceleration.

In [ ]:
# Colab setup — skip this cell if running locally
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import os
    import subprocess
    import getpass

    from google.colab import drive
    drive.mount("/content/drive")

    subprocess.run([
        "pip", "install", "-q",
        "torch", "torchvision", "transformers", "sentence-transformers",
        "datasets", "Pillow", "python-dotenv", "tqdm",
        "scikit-learn", "scipy", "pandas", "matplotlib", "seaborn",
    ], check=True)

    REPO_DIR = "/content/spatial_probing"
    if not os.path.exists(REPO_DIR):
        gh_token = getpass.getpass("GitHub Personal Access Token (ghp_...): ")
        clone_url = f"https://{gh_token}@github.com/nogully/spatial_probing.git"
        result = subprocess.run(
            ["git", "clone", clone_url, REPO_DIR],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f"git clone failed:\n{result.stderr}")
        print("Clone succeeded.")
    else:
        print(f"Repo already cloned at {REPO_DIR}, pulling latest...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

    os.environ["CACHE_DIR"] = "/content/drive/MyDrive/spatial_probing/results/embeddings"
    os.environ["HF_TOKEN"] = ""  # paste your HuggingFace token here

    print("Colab setup complete.")
else:
    print("Running locally — skipping Colab setup.")

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
from dotenv import load_dotenv
import os

PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

CACHE_DIR = Path(os.getenv("CACHE_DIR", "results/embeddings"))
if not CACHE_DIR.is_absolute():
    CACHE_DIR = PROJECT_ROOT / CACHE_DIR
CACHE_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
torch.manual_seed(42)

print(f"Project root: {PROJECT_ROOT}")
print(f"Cache directory: {CACHE_DIR}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 2. Load VSR Dataset

In [ ]:
import concurrent.futures
import requests
from PIL import Image as PILImage
from io import BytesIO
from datasets import load_dataset
from tqdm import tqdm

print("Loading VSR dataset...")
vsr = load_dataset("cambridgeltl/vsr_random")
train_data = vsr["train"]
print(f"VSR train set: {len(train_data)} examples")

# Extract texts
texts = [ex["caption"] for ex in train_data]

# VSR stores COCO filenames, not image bytes — download from COCO URLs
filenames = [ex["image"] for ex in train_data]
unique_filenames = list(set(filenames))
print(f"Unique COCO images to download: {len(unique_filenames)}")

def download_coco_image(filename):
    for split in ["train2017", "val2017"]:
        url = f"http://images.cocodataset.org/{split}/{filename}"
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200:
                return filename, PILImage.open(BytesIO(r.content)).convert("RGB")
        except Exception:
            pass
    return filename, None

print("Downloading images from COCO (parallel)...")
image_cache = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(download_coco_image, fn): fn for fn in unique_filenames}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(unique_filenames)):
        filename, img = future.result()
        image_cache[filename] = img

failed = sum(1 for img in image_cache.values() if img is None)
print(f"Downloaded: {len(image_cache) - failed}/{len(unique_filenames)}, Failed: {failed}")

images = [image_cache.get(fn) for fn in filenames]

print(f"\nSample captions:")
for i in range(3):
    print(f"  {i+1}. {texts[i]}")

## 3. Extract SBERT Embeddings

In [ ]:
from src.embedders import SBERTEmbedder

# Check cache
sbert_cache = CACHE_DIR / "sbert_vsr_train.npy"

if sbert_cache.exists():
    print(f"Loading cached SBERT embeddings from {sbert_cache}")
    sbert_embeddings = np.load(sbert_cache)
    print(f"Loaded shape: {sbert_embeddings.shape}")
else:
    print(f"Computing SBERT embeddings...")
    embedder = SBERTEmbedder()
    sbert_embeddings = embedder.embed_text(texts, batch_size=64)
    print(f"Extracted embeddings: {sbert_embeddings.shape}")
    
    # Verify L2 normalization
    norms = np.linalg.norm(sbert_embeddings, axis=1)
    print(f"Embedding norms (should be ~1.0): min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}")
    
    # Cache
    np.save(sbert_cache, sbert_embeddings.astype(np.float32))
    print(f"Saved to {sbert_cache}")

## 4. Extract CLIP Text Embeddings

In [ ]:
from src.embedders import CLIPTextEmbedder

# Check cache
clip_text_cache = CACHE_DIR / "clip_text_vsr_train.npy"

if clip_text_cache.exists():
    print(f"Loading cached CLIP text embeddings from {clip_text_cache}")
    clip_text_embeddings = np.load(clip_text_cache)
    print(f"Loaded shape: {clip_text_embeddings.shape}")
else:
    print(f"Computing CLIP text embeddings...")
    embedder = CLIPTextEmbedder()
    clip_text_embeddings = embedder.embed_text(texts, batch_size=32)
    print(f"Extracted embeddings: {clip_text_embeddings.shape}")
    
    # Verify L2 normalization
    norms = np.linalg.norm(clip_text_embeddings, axis=1)
    print(f"Embedding norms (should be ~1.0): min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}")
    
    # Cache
    np.save(clip_text_cache, clip_text_embeddings.astype(np.float32))
    print(f"Saved to {clip_text_cache}")

## 5. Extract CLIP Image-Only Embeddings

In [ ]:
from src.embedders import CLIPMultimodalEmbedder

embedder = CLIPMultimodalEmbedder()

# --- Image-only ---
clip_image_cache = CACHE_DIR / "clip_image_vsr_train.npy"

if clip_image_cache.exists():
    print(f"Loading cached CLIP image embeddings from {clip_image_cache}")
    clip_image_embeddings = np.load(clip_image_cache)
    print(f"Loaded shape: {clip_image_embeddings.shape}")
else:
    print("Computing CLIP image-only embeddings...")
    clip_image_embeddings = embedder.embed_image(images, batch_size=32)
    print(f"Extracted embeddings: {clip_image_embeddings.shape}")
    norms = np.linalg.norm(clip_image_embeddings, axis=1)
    print(f"Norms (should be ~1.0): min={norms.min():.4f}, max={norms.max():.4f}")
    np.save(clip_image_cache, clip_image_embeddings.astype(np.float32))
    print(f"Saved to {clip_image_cache}")

# --- Concat fusion (image + text) ---
clip_concat_cache = CACHE_DIR / "clip_concat_vsr_train.npy"

if clip_concat_cache.exists():
    print(f"\nLoading cached CLIP concat embeddings from {clip_concat_cache}")
    clip_concat_embeddings = np.load(clip_concat_cache)
    print(f"Loaded shape: {clip_concat_embeddings.shape}")
else:
    print("\nComputing CLIP concat (image+text) embeddings...")
    clip_concat_embeddings = embedder.embed_image_text(images, texts, batch_size=32, fusion="concat")
    print(f"Extracted embeddings: {clip_concat_embeddings.shape}")
    norms = np.linalg.norm(clip_concat_embeddings, axis=1)
    print(f"Norms (should be ~1.0): min={norms.min():.4f}, max={norms.max():.4f}")
    np.save(clip_concat_cache, clip_concat_embeddings.astype(np.float32))
    print(f"Saved to {clip_concat_cache}")

## 6. Sanity Checks

In [ ]:
print("Embedding Shapes:")
print(f"  SBERT:            {sbert_embeddings.shape}")
print(f"  CLIP text:        {clip_text_embeddings.shape}")
print(f"  CLIP image:       {clip_image_embeddings.shape}")
print(f"  CLIP concat:      {clip_concat_embeddings.shape}")

assert sbert_embeddings.shape[0] == len(texts)
assert clip_text_embeddings.shape[0] == len(texts)
assert clip_image_embeddings.shape[0] == len(texts)
assert clip_concat_embeddings.shape[0] == len(texts)
assert clip_image_embeddings.shape[1] == 512
assert clip_concat_embeddings.shape[1] == 1024

print(f"\nL2 Normalization Check:")
for name, emb in [
    ("SBERT", sbert_embeddings),
    ("CLIP text", clip_text_embeddings),
    ("CLIP image", clip_image_embeddings),
    ("CLIP concat", clip_concat_embeddings),
]:
    norms = np.linalg.norm(emb, axis=1)
    print(f"  {name:14s}: norms in [{norms.min():.4f}, {norms.max():.4f}]")

## 7. Save Metadata

In [ ]:
import json
from datetime import datetime

metadata = {
    "dataset": "vsr_random",
    "split": "train",
    "n_examples": len(texts),
    "extraction_date": datetime.now().isoformat(),
    "models": {
        "sbert": {
            "name": "sentence-transformers/all-mpnet-base-v2",
            "dim": int(sbert_embeddings.shape[1]),
            "file": "sbert_vsr_train.npy",
        },
        "clip_text": {
            "name": "openai/clip-vit-base-patch32",
            "type": "text-only",
            "dim": int(clip_text_embeddings.shape[1]),
            "file": "clip_text_vsr_train.npy",
        },
        "clip_image": {
            "name": "openai/clip-vit-base-patch32",
            "type": "image-only",
            "dim": int(clip_image_embeddings.shape[1]),
            "file": "clip_image_vsr_train.npy",
        },
        "clip_concat": {
            "name": "openai/clip-vit-base-patch32",
            "type": "image+text concat",
            "dim": int(clip_concat_embeddings.shape[1]),
            "file": "clip_concat_vsr_train.npy",
        },
    },
    "normalization": "L2",
}

metadata_path = CACHE_DIR / "metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved metadata to {metadata_path}")
print(json.dumps(metadata, indent=2))

## 8. Summary

In [ ]:
print("\n" + "="*70)
print("EMBEDDING EXTRACTION COMPLETE")
print("="*70)
print(f"\nDataset: VSR (vsr_random, train split)")
print(f"Examples: {len(texts)}")
print(f"\nCached embeddings:")
print(f"  ✓ SBERT (768d):          {CACHE_DIR / 'sbert_vsr_train.npy'}")
print(f"  ✓ CLIP text (512d):      {CACHE_DIR / 'clip_text_vsr_train.npy'}")
print(f"  ✓ CLIP image (512d):     {CACHE_DIR / 'clip_image_vsr_train.npy'}")
print(f"  ✓ CLIP concat (1024d):   {CACHE_DIR / 'clip_concat_vsr_train.npy'}")
print(f"\nFiles:")
for f in sorted(CACHE_DIR.glob("*.npy")):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"  {f.name:35s} {size_mb:8.1f} MB")
print(f"\nAll embeddings are L2-normalized (norms ~1.0)")
print("="*70)